 # 00 · Publicar catálogo de reglas DQ Bronze — YelpConnect

 **Dominio:** Data Quality  
 **Capa:** Bronze  
 **Tipo:** Gobierno / publicación de catálogo  
 **Destino:** `workspace.yelp_gov.dq_reglas`

 ---

 ## Propósito

 Publicar en el catálogo central de gobierno las reglas de
 calidad aplicables a la capa Bronze.

 Este notebook:

 - define las reglas DQ Bronze;
 - valida su contrato;
 - valida duplicados y parametrización;
 - publica mediante `MERGE`;
 - verifica el catálogo resultante.

 Este notebook **NO**:

 - crea schemas;
 - crea tablas de gobierno;
 - modifica tablas Bronze;
 - ejecuta reglas DQ;
 - aplica el Quality Gate.

 La infraestructura de gobierno debe existir previamente mediante:

 `00_setup/02_configurar_modelo_gobierno`

 ---

 ### Principio

 **El catálogo declara las reglas; el ejecutor las evalúa.**%md


In [0]:
# COMMAND ----------

# ============================================================
# 1. IMPORTS
# ============================================================

import json
import uuid

from pyspark.sql import functions as F



In [0]:

# COMMAND ----------

# ============================================================
# 2. CONFIGURACIÓN
# ============================================================

CATALOG = "workspace"

GOV_SCHEMA = "yelp_gov"

BRONZE_SCHEMA = "yelp_bronze"

LAYER = "bronze"

PIPELINE_NAME = (
    "E5_00_publicar_catalogo_reglas_dq_bronze"
)

PUBLICATION_ID = str(
    uuid.uuid4()
)


DQ_RULE_TABLE = (
    f"{CATALOG}."
    f"{GOV_SCHEMA}."
    "dq_reglas"
)


print("=" * 72)
print("YELPCONNECT - CATÁLOGO DQ BRONZE")
print("=" * 72)

print(
    f"Pipeline       : {PIPELINE_NAME}"
)

print(
    f"Publication ID : {PUBLICATION_ID}"
)

print(
    f"Layer          : {LAYER}"
)

print(
    f"Destino        : {DQ_RULE_TABLE}"
)

print("=" * 72)


# COMMAND ----------

 ## 1 · Precondición de gobierno

 La tabla `dq_reglas` debe haber sido creada previamente.

 Este notebook no intenta reparar ni crear infraestructura
 automáticamente. Si el contrato de gobierno no existe,
 la publicación debe fallar.

In [0]:
# COMMAND ----------

# ============================================================
# 3. VALIDAR EXISTENCIA DE dq_reglas
# ============================================================

if not spark.catalog.tableExists(
    DQ_RULE_TABLE
):

    raise RuntimeError(
        f"""
        ❌ PRECONDICIÓN DE GOBIERNO INCUMPLIDA

        No existe:
        {DQ_RULE_TABLE}

        Ejecutar previamente:

        00_setup/02_configurar_modelo_gobierno

        o:

        00_setup/04_orquestar_gobierno_catalogos
        """
    )


print(
    f"✔ Tabla de gobierno disponible: "
    f"{DQ_RULE_TABLE}"
)


In [0]:
# COMMAND ----------

# ============================================================
# 4. VALIDAR CONTRATO DE dq_reglas
# ============================================================

EXPECTED_CONTRACT = {

    "regla_id":
        "string",

    "capa":
        "string",

    "tabla_objetivo":
        "string",

    "columna":
        "string",

    "dimension":
        "string",

    "tipo_regla":
        "string",

    "descripcion":
        "string",

    "expresion_sql":
        "string",

    "criticidad":
        "string",

    "umbral_pct":
        "double"
}


actual_schema = {
    field.name:
        field.dataType.simpleString()
    for field
    in spark.table(
        DQ_RULE_TABLE
    ).schema.fields
}


missing_columns = [
    column_name
    for column_name
    in EXPECTED_CONTRACT
    if column_name
    not in actual_schema
]


if missing_columns:

    raise RuntimeError(
        f"""
        ❌ CONTRATO dq_reglas INVÁLIDO

        Columnas faltantes:

        {missing_columns}
        """
    )


invalid_types = []


for (
    column_name,
    expected_type
) in EXPECTED_CONTRACT.items():

    actual_type = (
        actual_schema[
            column_name
        ]
    )

    if (
        actual_type
        !=
        expected_type
    ):

        invalid_types.append(
            (
                column_name,
                expected_type,
                actual_type
            )
        )


if invalid_types:

    raise RuntimeError(
        f"""
        ❌ TIPOS INCOMPATIBLES EN dq_reglas

        Detalle:
        {invalid_types}
        """
    )


print(
    "✔ Contrato de dq_reglas validado."
)


# COMMAND ----------



 ## 2 · Catálogo canónico DQ Bronze

 Las reglas Bronze evalúan exclusivamente aspectos compatibles
 con una capa de conservación:

 - completitud;
 - unicidad;
 - validez técnica;
 - trazabilidad.

 No se realizan correcciones ni transformaciones semánticas.

 Las relaciones de negocio y controles más complejos
 corresponden a Silver.

In [0]:
# COMMAND ----------

# ============================================================
# 5. CONTRATO DE REGLAS
# ============================================================

RULE_SCHEMA = """
    regla_id STRING,
    capa STRING,
    tabla_objetivo STRING,
    columna STRING,
    dimension STRING,
    tipo_regla STRING,
    descripcion STRING,
    expresion_sql STRING,
    criticidad STRING,
    umbral_pct DOUBLE
"""


In [0]:

# COMMAND ----------

# ============================================================
# 6. DEFINICIÓN CANÓNICA DE REGLAS BRONZE
# ============================================================

reglas_bronze = [

    # ========================================================
    # BUSINESS
    # ========================================================

    (
        "DQ-BR-001",
        "bronze",
        "workspace.yelp_bronze.brz_business",
        "*",
        "completitud",
        "TABLE_NOT_EMPTY",
        "Business debe contener registros",
        None,
        "alta",
        100.0
    ),

    (
        "DQ-BR-002",
        "bronze",
        "workspace.yelp_bronze.brz_business",
        "business_id",
        "completitud",
        "CONDITION",
        "business_id debe estar informado",
        "business_id IS NOT NULL",
        "media",
        100.0
    ),

    (
        "DQ-BR-003",
        "bronze",
        "workspace.yelp_bronze.brz_business",
        "business_id",
        "unicidad",
        "UNIQUENESS",
        "business_id debe ser único",
        None,
        "media",
        100.0
    ),

    (
        "DQ-BR-004",
        "bronze",
        "workspace.yelp_bronze.brz_business",
        "_metadata",
        "trazabilidad",
        "CONDITION",
        "Business debe poseer metadata técnica completa de linaje",
        """
        _archivo_origen IS NOT NULL
        AND _fecha_ingesta IS NOT NULL
        AND _source_system = 'YELP'
        AND _source_entity = 'business'
        AND _pipeline_name IS NOT NULL
        AND _execution_id IS NOT NULL
        AND _layer = 'bronze'
        AND _schema_hash IS NOT NULL
        AND _record_checksum IS NOT NULL
        """,
        "alta",
        100.0
    ),


    # ========================================================
    # USER
    # ========================================================

    (
        "DQ-BR-005",
        "bronze",
        "workspace.yelp_bronze.brz_user",
        "*",
        "completitud",
        "TABLE_NOT_EMPTY",
        "User debe contener registros",
        None,
        "alta",
        100.0
    ),

    (
        "DQ-BR-006",
        "bronze",
        "workspace.yelp_bronze.brz_user",
        "user_id",
        "completitud",
        "CONDITION",
        "user_id debe estar informado",
        "user_id IS NOT NULL",
        "media",
        100.0
    ),

    (
        "DQ-BR-007",
        "bronze",
        "workspace.yelp_bronze.brz_user",
        "user_id",
        "unicidad",
        "UNIQUENESS",
        "user_id debe ser único",
        None,
        "media",
        100.0
    ),

    (
        "DQ-BR-008",
        "bronze",
        "workspace.yelp_bronze.brz_user",
        "_metadata",
        "trazabilidad",
        "CONDITION",
        "User debe poseer metadata técnica completa de linaje",
        """
        _archivo_origen IS NOT NULL
        AND _fecha_ingesta IS NOT NULL
        AND _source_system = 'YELP'
        AND _source_entity = 'user'
        AND _pipeline_name IS NOT NULL
        AND _execution_id IS NOT NULL
        AND _layer = 'bronze'
        AND _schema_hash IS NOT NULL
        AND _record_checksum IS NOT NULL
        """,
        "alta",
        100.0
    ),


    # ========================================================
    # REVIEW
    # ========================================================

    (
        "DQ-BR-009",
        "bronze",
        "workspace.yelp_bronze.brz_review",
        "*",
        "completitud",
        "TABLE_NOT_EMPTY",
        "Review debe contener registros",
        None,
        "alta",
        100.0
    ),

    (
        "DQ-BR-010",
        "bronze",
        "workspace.yelp_bronze.brz_review",
        "review_id",
        "completitud",
        "CONDITION",
        "review_id debe estar informado",
        "review_id IS NOT NULL",
        "media",
        100.0
    ),

    (
        "DQ-BR-011",
        "bronze",
        "workspace.yelp_bronze.brz_review",
        "review_id",
        "unicidad",
        "UNIQUENESS",
        "review_id debe ser único",
        None,
        "media",
        100.0
    ),

    (
        "DQ-BR-012",
        "bronze",
        "workspace.yelp_bronze.brz_review",
        "business_id",
        "completitud",
        "CONDITION",
        "Toda Review debe contener business_id",
        "business_id IS NOT NULL",
        "media",
        100.0
    ),

    (
        "DQ-BR-013",
        "bronze",
        "workspace.yelp_bronze.brz_review",
        "user_id",
        "completitud",
        "CONDITION",
        "Toda Review debe contener user_id",
        "user_id IS NOT NULL",
        "media",
        100.0
    ),

    (
        "DQ-BR-014",
        "bronze",
        "workspace.yelp_bronze.brz_review",
        "stars",
        "validez",
        "CONDITION",
        "El rating original debe encontrarse entre 1 y 5",
        "try_cast(stars AS DOUBLE) BETWEEN 1 AND 5",
        "media",
        100.0
    ),

    (
        "DQ-BR-015",
        "bronze",
        "workspace.yelp_bronze.brz_review",
        "_metadata",
        "trazabilidad",
        "CONDITION",
        "Review debe poseer metadata técnica completa de linaje",
        """
        _archivo_origen IS NOT NULL
        AND _fecha_ingesta IS NOT NULL
        AND _source_system = 'YELP'
        AND _source_entity = 'review'
        AND _pipeline_name IS NOT NULL
        AND _execution_id IS NOT NULL
        AND _layer = 'bronze'
        AND _schema_hash IS NOT NULL
        AND _record_checksum IS NOT NULL
        """,
        "alta",
        100.0
    ),


    # ========================================================
    # TIP
    # ========================================================

    (
        "DQ-BR-016",
        "bronze",
        "workspace.yelp_bronze.brz_tip",
        "*",
        "completitud",
        "TABLE_NOT_EMPTY",
        "Tip debe contener registros",
        None,
        "media",
        100.0
    ),

    (
        "DQ-BR-017",
        "bronze",
        "workspace.yelp_bronze.brz_tip",
        "business_id",
        "completitud",
        "CONDITION",
        "Tip debe contener business_id",
        "business_id IS NOT NULL",
        "media",
        100.0
    ),

    (
        "DQ-BR-018",
        "bronze",
        "workspace.yelp_bronze.brz_tip",
        "_metadata",
        "trazabilidad",
        "CONDITION",
        "Tip debe poseer metadata técnica completa de linaje",
        """
        _archivo_origen IS NOT NULL
        AND _fecha_ingesta IS NOT NULL
        AND _source_system = 'YELP'
        AND _source_entity = 'tip'
        AND _pipeline_name IS NOT NULL
        AND _execution_id IS NOT NULL
        AND _layer = 'bronze'
        AND _schema_hash IS NOT NULL
        AND _record_checksum IS NOT NULL
        """,
        "alta",
        100.0
    ),


    # ========================================================
    # CHECKIN
    # ========================================================

    (
        "DQ-BR-019",
        "bronze",
        "workspace.yelp_bronze.brz_checkin",
        "*",
        "completitud",
        "TABLE_NOT_EMPTY",
        "Check-in debe contener registros",
        None,
        "media",
        100.0
    ),

    (
        "DQ-BR-020",
        "bronze",
        "workspace.yelp_bronze.brz_checkin",
        "business_id",
        "completitud",
        "CONDITION",
        "Check-in debe contener business_id",
        "business_id IS NOT NULL",
        "media",
        100.0
    ),

    (
        "DQ-BR-021",
        "bronze",
        "workspace.yelp_bronze.brz_checkin",
        "date",
        "completitud",
        "CONDITION",
        "Check-in debe contener fechas",
        "date IS NOT NULL",
        "media",
        100.0
    ),

    (
        "DQ-BR-022",
        "bronze",
        "workspace.yelp_bronze.brz_checkin",
        "_metadata",
        "trazabilidad",
        "CONDITION",
        "Check-in debe poseer metadata técnica completa de linaje",
        """
        _archivo_origen IS NOT NULL
        AND _fecha_ingesta IS NOT NULL
        AND _source_system = 'YELP'
        AND _source_entity = 'checkin'
        AND _pipeline_name IS NOT NULL
        AND _execution_id IS NOT NULL
        AND _layer = 'bronze'
        AND _schema_hash IS NOT NULL
        AND _record_checksum IS NOT NULL
        """,
        "alta",
        100.0
    )
]


In [0]:
# COMMAND ----------

# ============================================================
# 7. CONSTRUIR DATAFRAME CANÓNICO
# ============================================================

df_reglas_bronze = (
    spark.createDataFrame(
        reglas_bronze,
        RULE_SCHEMA
    )
)


EXPECTED_RULE_COUNT = 22


actual_rule_count = (
    df_reglas_bronze.count()
)


if (
    actual_rule_count
    !=
    EXPECTED_RULE_COUNT
):

    raise RuntimeError(
        f"""
        ❌ CATÁLOGO BRONZE INCOMPLETO

        Esperadas:
        {EXPECTED_RULE_COUNT}

        Definidas:
        {actual_rule_count}
        """
    )


print(
    f"✔ Reglas declaradas: "
    f"{actual_rule_count}"
)


# COMMAND ----------


 ## 3 · Validación del catálogo

 Antes de publicar se comprueba que el conjunto de reglas
 sea internamente consistente.

In [0]:
# COMMAND ----------

# ============================================================
# 8. VALIDAR IDs DUPLICADOS
# ============================================================

duplicate_ids = (
    df_reglas_bronze

    .groupBy(
        "regla_id"
    )

    .count()

    .filter(
        F.col("count") > 1
    )

    .collect()
)


if duplicate_ids:

    raise RuntimeError(
        f"""
        ❌ EXISTEN regla_id DUPLICADOS

        {duplicate_ids}
        """
    )


print(
    "✔ regla_id sin duplicados."
)



In [0]:
# COMMAND ----------

# ============================================================
# 9. VALIDAR CAPA
# ============================================================

invalid_layers = (
    df_reglas_bronze

    .filter(
        F.col("capa") != LAYER
    )

    .count()
)


if invalid_layers > 0:

    raise RuntimeError(
        f"""
        ❌ Se detectaron reglas fuera
        de la capa {LAYER}.
        """
    )


print(
    "✔ Todas las reglas pertenecen a Bronze."
)


In [0]:
# COMMAND ----------

# ============================================================
# 10. VALIDAR DIMENSIONES
# ============================================================

ALLOWED_DIMENSIONS = {
    "completitud",
    "unicidad",
    "validez",
    "trazabilidad"
}


invalid_dimensions = (
    df_reglas_bronze

    .filter(
        ~F.col(
            "dimension"
        ).isin(
            list(
                ALLOWED_DIMENSIONS
            )
        )
    )

    .select(
        "regla_id",
        "dimension"
    )

    .collect()
)


if invalid_dimensions:

    raise RuntimeError(
        f"""
        ❌ DIMENSIONES DQ NO SOPORTADAS

        {invalid_dimensions}
        """
    )


print(
    "✔ Dimensiones DQ válidas."
)


In [0]:
# COMMAND ----------

# ============================================================
# 11. VALIDAR TIPOS DE REGLA
# ============================================================

ALLOWED_RULE_TYPES = {
    "TABLE_NOT_EMPTY",
    "CONDITION",
    "UNIQUENESS"
}


invalid_types = (
    df_reglas_bronze

    .filter(
        ~F.col(
            "tipo_regla"
        ).isin(
            list(
                ALLOWED_RULE_TYPES
            )
        )
    )

    .select(
        "regla_id",
        "tipo_regla"
    )

    .collect()
)


if invalid_types:

    raise RuntimeError(
        f"""
        ❌ TIPOS DE REGLA NO SOPORTADOS

        {invalid_types}
        """
    )


print(
    "✔ Tipos de regla válidos."
)


In [0]:
# COMMAND ----------

# ============================================================
# 12. VALIDAR CRITICIDAD
# ============================================================

ALLOWED_CRITICALITY = {
    "alta",
    "media",
    "baja"
}


invalid_criticality = (
    df_reglas_bronze

    .filter(
        ~F.col(
            "criticidad"
        ).isin(
            list(
                ALLOWED_CRITICALITY
            )
        )
    )

    .select(
        "regla_id",
        "criticidad"
    )

    .collect()
)


if invalid_criticality:

    raise RuntimeError(
        f"""
        ❌ CRITICIDAD NO SOPORTADA

        {invalid_criticality}
        """
    )


print(
    "✔ Criticidad válida."
)


In [0]:
# COMMAND ----------

# ============================================================
# 13. VALIDAR UMBRALES
# ============================================================

invalid_thresholds = (
    df_reglas_bronze

    .filter(
        F.col(
            "umbral_pct"
        ).isNull()

        |

        (
            F.col(
                "umbral_pct"
            )
            <
            0.0
        )

        |

        (
            F.col(
                "umbral_pct"
            )
            >
            100.0
        )
    )

    .select(
        "regla_id",
        "umbral_pct"
    )

    .collect()
)


if invalid_thresholds:

    raise RuntimeError(
        f"""
        ❌ UMBRALES DQ INVÁLIDOS

        {invalid_thresholds}
        """
    )


print(
    "✔ Umbrales válidos."
)


In [0]:
# COMMAND ----------

# ============================================================
# 14. VALIDAR EXPRESIONES CONDITION
# ============================================================

invalid_conditions = (
    df_reglas_bronze

    .filter(
        (
            F.col(
                "tipo_regla"
            )
            ==
            "CONDITION"
        )

        &

        (
            F.col(
                "expresion_sql"
            ).isNull()

            |

            (
                F.length(
                    F.trim(
                        F.col(
                            "expresion_sql"
                        )
                    )
                )
                ==
                0
            )
        )
    )

    .select(
        "regla_id"
    )

    .collect()
)


if invalid_conditions:

    raise RuntimeError(
        f"""
        ❌ CONDITION SIN expresion_sql

        {invalid_conditions}
        """
    )


print(
    "✔ CONDITION posee expresion_sql."
)


In [0]:


# COMMAND ----------

# ============================================================
# 15. VALIDAR UNIQUENESS
# ============================================================

invalid_uniqueness = (
    df_reglas_bronze

    .filter(
        (
            F.col(
                "tipo_regla"
            )
            ==
            "UNIQUENESS"
        )

        &

        (
            F.col(
                "columna"
            ).isNull()

            |

            (
                F.col(
                    "columna"
                )
                ==
                "*"
            )
        )
    )

    .select(
        "regla_id",
        "columna"
    )

    .collect()
)


if invalid_uniqueness:

    raise RuntimeError(
        f"""
        ❌ UNIQUENESS SIN COLUMNA VÁLIDA

        {invalid_uniqueness}
        """
    )


print(
    "✔ Reglas UNIQUENESS válidas."
)


# COMMAND ----------

 ## 4 · Publicación

 La publicación utiliza `MERGE` por `regla_id`.

 De esta manera:

 - una regla existente actualiza su definición;
 - una regla nueva se incorpora;
 - no se generan duplicados.


In [0]:


# COMMAND ----------

# ============================================================
# 16. PUBLICAR CATÁLOGO
# ============================================================

df_reglas_bronze.createOrReplaceTempView(
    "src_dq_reglas_bronze"
)


spark.sql(
    f"""
    MERGE INTO
        {DQ_RULE_TABLE} AS target

    USING
        src_dq_reglas_bronze AS source

    ON
        target.regla_id = source.regla_id

    WHEN MATCHED THEN
        UPDATE SET

            target.capa =
                source.capa,

            target.tabla_objetivo =
                source.tabla_objetivo,

            target.columna =
                source.columna,

            target.dimension =
                source.dimension,

            target.tipo_regla =
                source.tipo_regla,

            target.descripcion =
                source.descripcion,

            target.expresion_sql =
                source.expresion_sql,

            target.criticidad =
                source.criticidad,

            target.umbral_pct =
                source.umbral_pct

    WHEN NOT MATCHED THEN
        INSERT
        (
            regla_id,
            capa,
            tabla_objetivo,
            columna,
            dimension,
            tipo_regla,
            descripcion,
            expresion_sql,
            criticidad,
            umbral_pct
        )

        VALUES
        (
            source.regla_id,
            source.capa,
            source.tabla_objetivo,
            source.columna,
            source.dimension,
            source.tipo_regla,
            source.descripcion,
            source.expresion_sql,
            source.criticidad,
            source.umbral_pct
        )
    """
)


print(
    "✔ MERGE del catálogo Bronze completado."
)


# COMMAND ----------

 ## 5 · Validación posterior a la publicación


In [0]:

# COMMAND ----------

# ============================================================
# 17. VALIDAR IDs PUBLICADOS
# ============================================================

expected_ids = {
    row.regla_id
    for row
    in df_reglas_bronze
        .select(
            "regla_id"
        )
        .collect()
}


published_ids = {
    row.regla_id
    for row
    in (
        spark
        .table(
            DQ_RULE_TABLE
        )
        .filter(
            F.col(
                "capa"
            )
            ==
            LAYER
        )
        .select(
            "regla_id"
        )
        .collect()
    )
}


missing_ids = (
    expected_ids
    -
    published_ids
)


if missing_ids:

    raise RuntimeError(
        f"""
        ❌ REGLAS BRONZE NO PUBLICADAS

        {sorted(missing_ids)}
        """
    )


print(
    "✔ Las 22 reglas esperadas "
    "se encuentran publicadas."
)


In [0]:

# COMMAND ----------

# ============================================================
# 18. DETECTAR REGLAS BRONZE NO CANÓNICAS
# ============================================================

extra_ids = (
    published_ids
    -
    expected_ids
)


if extra_ids:

    raise RuntimeError(
        f"""
        ❌ CATÁLOGO BRONZE CONTIENE REGLAS NO CANÓNICAS

        Reglas adicionales:

        {sorted(extra_ids)}

        Revisar antes de continuar.

        No se eliminan automáticamente para evitar
        borrar definiciones de gobierno sin aprobación.
        """
    )


print(
    "✔ No existen reglas Bronze adicionales."
)

In [0]:

# COMMAND ----------

# ============================================================
# 19. VALIDAR TOTAL FINAL
# ============================================================

bronze_rule_count = (

    spark

    .table(
        DQ_RULE_TABLE
    )

    .filter(
        F.col(
            "capa"
        )
        ==
        LAYER
    )

    .count()
)


if (
    bronze_rule_count
    !=
    EXPECTED_RULE_COUNT
):

    raise RuntimeError(
        f"""
        ❌ TOTAL DE REGLAS BRONZE INCORRECTO

        Esperadas:
        {EXPECTED_RULE_COUNT}

        Publicadas:
        {bronze_rule_count}
        """
    )


print(
    f"✔ Total catálogo Bronze: "
    f"{bronze_rule_count}"
)


 ## 6 · Inventario publicado

In [0]:
# COMMAND ----------

# ============================================================
# 20. MOSTRAR CATÁLOGO
# ============================================================

display(
    spark
    .table(
        DQ_RULE_TABLE
    )
    .filter(
        F.col(
            "capa"
        )
        ==
        LAYER
    )
    .orderBy(
        "regla_id"
    )
)


In [0]:


# COMMAND ----------

# ============================================================
# 21. RESUMEN POR DIMENSIÓN
# ============================================================

display(
    spark
    .table(
        DQ_RULE_TABLE
    )
    .filter(
        F.col(
            "capa"
        )
        ==
        LAYER
    )
    .groupBy(
        "dimension"
    )
    .agg(

        F.count(
            "*"
        ).alias(
            "reglas"
        ),

        F.sum(
            F.when(
                F.col(
                    "criticidad"
                )
                ==
                "alta",
                1
            )
            .otherwise(
                0
            )
        ).alias(
            "criticidad_alta"
        ),

        F.sum(
            F.when(
                F.col(
                    "criticidad"
                )
                ==
                "media",
                1
            )
            .otherwise(
                0
            )
        ).alias(
            "criticidad_media"
        )
    )
    .orderBy(
        "dimension"
    )
)


# COMMAND ----------


 ## Definition of Done

 - [x] Infraestructura de gobierno validada.
 - [x] Contrato `dq_reglas` validado.
 - [x] 22 reglas Bronze declaradas.
 - [x] IDs únicos.
 - [x] Dimensiones controladas.
 - [x] Tipos de regla controlados.
 - [x] Criticidad controlada.
 - [x] Umbrales validados.
 - [x] Catálogo publicado mediante MERGE.
 - [x] Catálogo final reconciliado.
 - [x] Ningún dato Bronze fue modificado.


In [0]:

# COMMAND ----------

# ============================================================
# 22. RESULTADO PARA ORQUESTADOR
# ============================================================

summary = {

    "status":
        "SUCCESS",

    "pipeline":
        PIPELINE_NAME,

    "publication_id":
        PUBLICATION_ID,

    "layer":
        LAYER,

    "rules_published":
        bronze_rule_count,

    "rule_table":
        DQ_RULE_TABLE
}


print()
print("=" * 72)
print("PUBLICACIÓN CATÁLOGO DQ BRONZE")
print("=" * 72)

print(
    json.dumps(
        summary,
        indent=2
    )
)

print("=" * 72)


dbutils.notebook.exit(
    json.dumps(
        summary
    )
)